# Fig. 6: training loss

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig01"]
seed_everything(cfg["seed"])
record_run("Fig01", cfg)


In [ ]:
# =====================================================
# One-pass deterministic batch-loss comparison
# Model-based (once) vs Unfolded (zero / no-zero)
# =====================================================

import os
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader, random_split

from solutions.trihybrid_WMMSE import GNN, TriHybridWMMSE
from solutions.utils import channelDataset, init_kaiming, zero_last_linear

# =====================================================
# Deterministic CPU setup
# =====================================================
BASE_SEED = cfg["seed"]
torch.manual_seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.use_deterministic_algorithms(True)

cdtype = torch.cfloat

# =====================================================
# Configuration
# =====================================================
BATCH_SIZE = cfg["batch_size"]
I_max = cfg["layers"]
LR = cfg["learning_rate"]
MAX_TRACK_BATCH = cfg["max_batches"]

RESULT_DIR = result_dir("results_BatchLoss")
os.makedirs(RESULT_DIR, exist_ok=True)

# =====================================================
# Load parameters & channels
# =====================================================
SAVE_DIR = DATA_DIR
parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))
channels = load_artifact(os.path.join(SAVE_DIR, "channels_deepMIMO_train.pt"))

Nt = parameters["Nt"]
Nc = parameters["Nc"]
P = parameters["P"]
Mk = parameters["Mk"]
eta = parameters["eta"]
Ns_k = parameters["Ns_k"]
sigma2 = parameters["sigma2"]

# =====================================================
# Generate deterministic problem instances
# =====================================================
num_ProbInsts = cfg["instances"]
ProbInsts = []

for s_idx in tqdm(range(num_ProbInsts), desc="Generating Problem Instances"):
    g = torch.Generator().manual_seed(BASE_SEED + s_idx)
    K = torch.randint(3, 6, (1,), generator=g).item()
    sel = torch.randperm(channels.shape[0], generator=g)[:K]
    Hsel = torch.tensor(channels[sel], dtype=cdtype).squeeze(-1)
    H = [Hsel[k] for k in range(K)]
    ProbInsts.append({"H": H, "K": K})

# =====================================================
# Dataset & fixed batches
# =====================================================
dataset = channelDataset(ProbInsts, parameters)
num_test = int(0.0025 * len(dataset))
num_train = len(dataset) - num_test

g_split = torch.Generator().manual_seed(BASE_SEED)
train_dataset, _ = random_split(dataset, [num_train, num_test], generator=g_split)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,                 # <<< critical
    collate_fn=lambda x: x
)

# =====================================================
# Build TWO unfolded models (zero / no-zero)
# =====================================================
in_dim = 2 * ProbInsts[0]["H"][0].numel() + 1

def build_unfolded(use_zero_last: bool):
    Gg, Gu = torch.nn.ModuleList(), torch.nn.ModuleList()
    for _ in range(I_max):
        g_gamma = GNN(
            in_dim=in_dim,
            hidden_dim=2 * Ns_k**2 + 4,
            out_dim=2 * Ns_k**2,
            num_layers=3,
            dp_rate=0.1
        )
        g_u = GNN(
            in_dim=in_dim,
            hidden_dim=Nt + 10,
            out_dim=Nt,
            num_layers=3,
            dp_rate=0.1
        )
        g_gamma.apply(init_kaiming)
        g_u.apply(init_kaiming)
        if use_zero_last:
            zero_last_linear(g_gamma)
            zero_last_linear(g_u)
        Gg.append(g_gamma)
        Gu.append(g_u)
    return Gg, Gu

GNNs_gamma_zero, GNNs_u_zero = build_unfolded(True)
GNNs_gamma_no,   GNNs_u_no   = build_unfolded(False)

# =====================================================
# Optimizers (independent)
# =====================================================
opt_zero = torch.optim.Adam(
    list(p for g in GNNs_gamma_zero for p in g.parameters()) +
    list(p for g in GNNs_u_zero for p in g.parameters()),
    lr=LR
)

opt_no = torch.optim.Adam(
    list(p for g in GNNs_gamma_no for p in g.parameters()) +
    list(p for g in GNNs_u_no for p in g.parameters()),
    lr=LR
)

# =====================================================
# Storage
# =====================================================
loss_mb = []
loss_zero = []
loss_no = []

# =====================================================
# Main loop (baseline ONCE)
# =====================================================
for bidx, batch in enumerate(train_loader):
    if bidx >= MAX_TRACK_BATCH:
        break

    opt_zero.zero_grad()
    opt_no.zero_grad()

    sum_loss_zero = 0.0
    sum_loss_no = 0.0
    sum_rate_mb = 0.0

    for inst in batch:
        K = inst["K"]
        node_features = inst["node_feature"]
        edge_index = inst["edge_index"]
        H_list = inst["H_list"]

        # ----- shared initialization -----
        u0 = [torch.ones(Nc, 1, dtype=cdtype) for _ in range(Nt)]
        v0 = [torch.ones(Ns_k * K, 1, dtype=cdtype) for _ in range(Nt)]
        for n in range(Nt):
            nv = torch.linalg.norm(v0[n])
            if nv > torch.sqrt(P):
                v0[n] = v0[n] / nv * torch.sqrt(P)

        # ===== Model-based (ONCE) =====
        solver_mb = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=[u.clone() for u in u0],
            v_init=[v.clone() for v in v0],
        )
        rate_mb, *_ = solver_mb.run_classical(I_max=I_max, verbose=False)
        sum_rate_mb += rate_mb[-1]

        # ===== Unfolded (zero-last) =====
        dgamma, du = [], []
        for i in range(I_max):
            dgamma.append(GNNs_gamma_zero[i](node_features, edge_index))
            du.append(
                GNNs_u_zero[i](node_features, edge_index,
                               torch.zeros(K, dtype=torch.int32),
                               torch.tensor(1, dtype=torch.int32)).reshape(Nt, 1)
            )

        solver_zero = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=[u.clone() for u in u0],
            v_init=[v.clone() for v in v0],
        )
        rate_z, *_ = solver_zero.run_unfolded(dgamma, du, I_max, False)
        sum_loss_zero += -rate_z[-1]

        # ===== Unfolded (no-zero) =====
        dgamma, du = [], []
        for i in range(I_max):
            dgamma.append(GNNs_gamma_no[i](node_features, edge_index))
            du.append(
                GNNs_u_no[i](node_features, edge_index,
                             torch.zeros(K, dtype=torch.int32),
                             torch.tensor(1, dtype=torch.int32)).reshape(Nt, 1)
            )

        solver_no = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=[u.clone() for u in u0],
            v_init=[v.clone() for v in v0],
        )
        rate_n, *_ = solver_no.run_unfolded(dgamma, du, I_max, False)
        sum_loss_no += -rate_n[-1]

    # ----- backward -----
    loss_zero_batch = sum_loss_zero / len(batch)
    loss_no_batch = sum_loss_no / len(batch)

    loss_zero_batch.backward()
    loss_no_batch.backward()

    opt_zero.step()
    opt_no.step()

    loss_mb.append(-(sum_rate_mb / len(batch)).item())
    loss_zero.append(loss_zero_batch.item())
    loss_no.append(loss_no_batch.item())

    print(
        f"[Batch {bidx:03d}] "
        f"MB={loss_mb[-1]:.4f}, "
        f"Zero={loss_zero[-1]:.4f}, "
        f"NoZero={loss_no[-1]:.4f}"
    )

# =====================================================
# Save
# =====================================================
np.save(os.path.join(RESULT_DIR, "loss_model_based.npy"), loss_mb)
np.save(os.path.join(RESULT_DIR, "loss_unfolded_zero.npy"), loss_zero)
np.save(os.path.join(RESULT_DIR, "loss_unfolded_nozero.npy"), loss_no)

print(f"\nSaved results to ./{RESULT_DIR}/")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

RESULT_DIR = result_dir("results_BatchLoss")

# =========================
# Load results
# =========================
loss_mb = np.load(os.path.join(RESULT_DIR, "loss_model_based.npy"))
loss_zero = np.load(os.path.join(RESULT_DIR, "loss_unfolded_zero.npy"))
loss_nozero = np.load(os.path.join(RESULT_DIR, "loss_unfolded_nozero.npy"))

num_batches = len(loss_mb)
x = np.arange(num_batches)

# =========================
# Plot
# =========================
plt.figure(figsize=(6, 4))

plt.plot(x, loss_mb, 
         linewidth=2,
         linestyle='--',
         color='black',
         label="Model-based WMMSE")

plt.plot(x, loss_nozero,
         linewidth=2,
         color='green',
         label="Unfolded WMMSE (no-zero)")

plt.plot(x, loss_zero,
         linewidth=2,
         color='red',
         label="Unfolded WMMSE (zero-last)")

plt.xlabel("Batch index")
plt.ylabel("Loss = −Average sum-rate")
plt.title("Batch-wise training loss comparison")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import os

RESULT_DIR = result_dir("results_BatchLoss")

# =========================
# Load results
# =========================
loss_mb = np.load(os.path.join(RESULT_DIR, "loss_model_based.npy"))
loss_zero = np.load(os.path.join(RESULT_DIR, "loss_unfolded_zero.npy"))
loss_nozero = np.load(os.path.join(RESULT_DIR, "loss_unfolded_nozero.npy"))

num_batches = len(loss_mb)
x = np.arange(num_batches)

curves = [
    ("Model-based WMMSE", x, loss_mb, "black", "--"),
    ("Unfolded WMMSE (no-zero)", x, loss_nozero, "green", "-"),
    ("Unfolded WMMSE (zero-last)", x, loss_zero, "red", "-"),
]

# =========================
# Print TikZ tables
# =========================
for label, xdata, ydata, color, linestyle in curves:
    print(r"\addplot[")
    print(r"    line width=1pt,")
    if linestyle == "--":
        print(r"    dashed,")
    print(f"    color={color},")
    print(r"]table[row sep=crcr]{%")

    for xi, yi in zip(xdata, ydata):
        print(f"{xi}\t{yi}\\\\")
    print("};")
    print(f"\\addlegendentry{{{label}}}")
    print("\n")